# Python 标准库工具

这个 notebook 汇总常用标准库工具，并补充应用级定时任务工具 APScheduler。章节从一级标题开始，API 场景使用二级/三级标题，便于 Jupyter、VS Code 和文档站点生成导航。


## APScheduler

APScheduler 是应用级定时任务工具，适合后台同步、报表生成、定时清理和轻量任务编排。它支持 interval、date、cron 等触发器，以及线程池/进程池执行器。


### 基本配置


In [ ]:

from apscheduler.executors.pool import ProcessPoolExecutor, ThreadPoolExecutor
from apscheduler.jobstores.memory import MemoryJobStore
from apscheduler.schedulers.background import BackgroundScheduler

scheduler = BackgroundScheduler(
    timezone="Asia/Shanghai",
    jobstores={"default": MemoryJobStore()},
    executors={
        "default": ThreadPoolExecutor(max_workers=10),
        "processpool": ProcessPoolExecutor(max_workers=2),
    },
    job_defaults={
        "coalesce": True,
        "max_instances": 1,
        "misfire_grace_time": 30,
    },
)


### Trigger 示例


In [ ]:

from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

from apscheduler.triggers.cron import CronTrigger
from apscheduler.triggers.date import DateTrigger
from apscheduler.triggers.interval import IntervalTrigger

SHANGHAI = ZoneInfo("Asia/Shanghai")


def send_report(report_name: str) -> str:
    """发送报表。

    Args:
        report_name: 报表名称。

    Returns:
        发送完成后的状态文本。
    """
    return f"{report_name} sent at {datetime.now(SHANGHAI):%Y-%m-%d %H:%M:%S}"


scheduler.add_job(
    send_report,
    trigger=IntervalTrigger(seconds=10),
    args=["interval-report"],
    id="interval_report",
    replace_existing=True,
)

scheduler.add_job(
    send_report,
    trigger=DateTrigger(run_date=datetime.now(SHANGHAI) + timedelta(minutes=5)),
    args=["once-report"],
    id="once_report",
    replace_existing=True,
)

scheduler.add_job(
    send_report,
    trigger=CronTrigger(hour=9, minute=30, timezone=SHANGHAI),
    args=["daily-report"],
    id="daily_report",
    replace_existing=True,
)


### 管理任务


In [ ]:
def refresh_cache(cache_name: str) -> str:
    """刷新缓存。

    Args:
        cache_name: 缓存名称。

    Returns:
        刷新结果。
    """
    return f"{cache_name} refreshed"


scheduler.add_job(refresh_cache, "interval", seconds=30, args=["user_profile"], id="refresh_cache")
job_ref = scheduler.get_job("refresh_cache")
print("Job:", job_ref)

scheduler.modify_job("refresh_cache", seconds=60)
scheduler.pause_job("refresh_cache")
scheduler.resume_job("refresh_cache")
scheduler.remove_job("refresh_cache")


### 监听器与生命周期


In [ ]:

from apscheduler.events import EVENT_JOB_ERROR, EVENT_JOB_EXECUTED, JobExecutionEvent


def log_job_event(event: JobExecutionEvent) -> None:
    """记录任务执行结果。

    Args:
        event: APScheduler 任务执行事件。
    """
    if event.exception:
        print(f"Job failed: {event.job_id}")
    else:
        print(f"Job succeeded: {event.job_id}")


scheduler.add_listener(log_job_event, EVENT_JOB_EXECUTED | EVENT_JOB_ERROR)
scheduler.print_jobs()

if not scheduler.running:
    scheduler.start(paused=True)

scheduler.resume()
scheduler.pause()
scheduler.shutdown(wait=False)


### 应用级封装


In [ ]:

from collections.abc import Callable

from apscheduler.schedulers.background import BackgroundScheduler


class SchedulerService:
    """应用中的调度服务封装。"""

    def __init__(self, scheduler: BackgroundScheduler) -> None:
        """初始化调度服务。

        Args:
            scheduler: APScheduler 后台调度器。
        """
        self._scheduler = scheduler

    def add_interval_job(
        self,
        job_id: str,
        func: Callable[..., object],
        seconds: int,
        *args: object,
    ) -> None:
        """注册可重复执行的 interval 任务。

        Args:
            job_id: 任务唯一 ID。
            func: 需要执行的函数。
            seconds: 执行间隔秒数。
            *args: 传给任务函数的位置参数。
        """
        self._scheduler.add_job(
            func,
            trigger="interval",
            seconds=seconds,
            args=args,
            id=job_id,
            replace_existing=True,
        )

    def start(self) -> None:
        """启动调度器。"""
        if not self._scheduler.running:
            self._scheduler.start()

    def stop(self) -> None:
        """停止调度器。"""
        if self._scheduler.running:
            self._scheduler.shutdown(wait=False)


## 数值计算工具

这一组标准库工具覆盖四类常见需求：实数数学函数、复数计算、十进制定点计算和有理数计算。工程里优先根据数据语义选工具，而不是统一用 `float`。


### `math`

`math` 面向实数计算，提供三角函数、组合数、最大公约数、精确求和、距离计算等能力。它适合普通工程计算，但涉及金额和精确小数时不要用 `float` 直接计算。


In [ ]:
import math


def distance(point_a: tuple[float, ...], point_b: tuple[float, ...]) -> float:
    """计算两个点之间的欧氏距离。

    Args:
        point_a: 第一个点的坐标。
        point_b: 第二个点的坐标。

    Returns:
        两个点之间的欧氏距离。
    """
    return math.dist(point_a, point_b)


def stable_average(values: list[float]) -> float:
    """使用更稳定的方式计算平均值。

    Args:
        values: 待统计的浮点数列表。

    Returns:
        浮点数平均值。

    Raises:
        ValueError: 输入列表为空。
    """
    if not values:
        raise ValueError("values must not be empty")
    return math.fsum(values) / len(values)


print("hypot:", math.hypot(3, 4))
print("distance:", distance((0, 0), (3, 4)))
print("average:", stable_average([0.1, 0.2, 0.3]))
print("is close:", math.isclose(0.1 + 0.2, 0.3, rel_tol=1e-9))
print("gcd/lcm:", math.gcd(12, 18), math.lcm(12, 18))
print("comb/perm:", math.comb(5, 2), math.perm(5, 2))


#### `math` 常用选择

| 需求 | 推荐 API | 说明 |
| --- | --- | --- |
| 判断浮点数近似相等 | `math.isclose()` | 不要直接用 `==` 比较浮点计算结果 |
| 多个浮点数求和 | `math.fsum()` | 比内置 `sum()` 更适合浮点误差敏感场景 |
| 坐标距离 | `math.dist()` / `math.hypot()` | 比手写平方和更清晰 |
| 组合排列 | `math.comb()` / `math.perm()` | 不需要自己写阶乘公式 |
| 整数约分辅助 | `math.gcd()` / `math.lcm()` | 常用于分数、周期和步长计算 |


### `cmath`

`cmath` 面向复数计算。即使输入是普通数字，它的返回值也通常是复数。信号处理、阻抗计算、复平面旋转和复数方程根可以优先考虑它。


In [ ]:
import cmath
import math as real_math


def impedance(resistance: float, reactance: float) -> complex:
    """构造电路阻抗的复数表示。

    Args:
        resistance: 电阻部分。
        reactance: 电抗部分。

    Returns:
        形如 ``resistance + reactance*j`` 的复数。
    """
    return complex(resistance, reactance)


def rotate(point: complex, radians: float) -> complex:
    """在复平面上旋转一个点。

    Args:
        point: 复平面上的点。
        radians: 旋转弧度。

    Returns:
        旋转后的复数坐标。
    """
    return point * cmath.exp(1j * radians)


z = impedance(3, 4)
radius, angle = cmath.polar(z)
restored = cmath.rect(radius, angle)
roots = [cmath.exp(2j * real_math.pi * index / 3) for index in range(3)]

print("z:", z)
print("abs:", abs(z))
print("phase:", cmath.phase(z))
print("polar/restored:", (radius, angle), restored)
print("rotated:", rotate(1 + 0j, real_math.pi / 2))
print("third roots of 1:", roots)


### `decimal`

`decimal.Decimal` 适合金额、结算、报表和需要明确舍入规则的小数计算。关键原则是：从字符串创建 `Decimal`，不要从浮点字面量创建。


In [ ]:
from decimal import Decimal, ROUND_HALF_UP, localcontext

CENT = Decimal("0.01")


def money(value: str) -> Decimal:
    """把字符串金额转换为两位小数。

    Args:
        value: 字符串形式的金额。

    Returns:
        按四舍五入规则保留两位小数的金额。
    """
    return Decimal(value).quantize(CENT, rounding=ROUND_HALF_UP)


def apply_tax(amount: Decimal, tax_rate: Decimal) -> Decimal:
    """计算含税金额。

    Args:
        amount: 未税金额。
        tax_rate: 税率，例如 ``Decimal("0.06")``。

    Returns:
        保留两位小数的含税金额。
    """
    return (amount * (Decimal("1") + tax_rate)).quantize(CENT, rounding=ROUND_HALF_UP)


subtotal = sum((money("19.995"), money("5.235"), money("3.10")), Decimal("0.00"))
total = apply_tax(subtotal, Decimal("0.06"))

with localcontext() as ctx:
    ctx.prec = 8
    ratio = Decimal("1") / Decimal("7")

print("float issue:", 0.1 + 0.2)
print("decimal sum:", Decimal("0.1") + Decimal("0.2"))
print("subtotal:", subtotal)
print("total:", total)
print("local precision:", ratio)


### `fractions.Fraction`

`Fraction` 表示有理数，适合比例、精确分数、配方换算和需要避免浮点误差的计算。它会自动约分，也可以用 `limit_denominator()` 把小数近似成易读分数。


In [ ]:
from fractions import Fraction


def completion_ratio(done: int, total: int) -> Fraction:
    """计算完成比例。

    Args:
        done: 已完成数量。
        total: 总数量。

    Returns:
        自动约分后的完成比例。

    Raises:
        ValueError: 总数量小于等于零。
    """
    if total <= 0:
        raise ValueError("total must be positive")
    return Fraction(done, total)


one_third = Fraction(1, 3)
recipe = {"flour": Fraction(3, 2), "sugar": Fraction(1, 4)}
scaled_recipe = {name: amount * 2 for name, amount in recipe.items()}
approx_pi = Fraction(3.1415926).limit_denominator(1000)

print("1/3 + 1/6:", one_third + Fraction(1, 6))
print("completion:", completion_ratio(45, 120))
print("scaled recipe:", scaled_recipe)
print("decimal string:", Fraction("0.125"))
print("pi approx:", approx_pi)


### `calendar`

`calendar` 用于生成和分析日历数据，适合月历展示、闰年判断、按周遍历日期、统计工作日和生成简单 HTML 日历。它只处理日历结构，不负责时区、时间点和夏令时；这些场景应结合 `datetime` 或 `zoneinfo`。


In [ ]:
import calendar
from datetime import date


def month_overview(year: int, month: int) -> dict[str, int | bool]:
    """获取月份基础信息。

    Args:
        year: 年份。
        month: 月份，范围是 1 到 12。

    Returns:
        包含首日星期、当月天数和是否闰年的字典。
    """
    first_weekday, days_in_month = calendar.monthrange(year, month)
    return {
        "first_weekday": first_weekday,
        "days_in_month": days_in_month,
        "is_leap_year": calendar.isleap(year),
    }


print(calendar.month(2026, 5))
print("Overview:", month_overview(2026, 5))
print("Leap days 2020-2030:", calendar.leapdays(2020, 2031))
print("Today weekday:", calendar.day_name[date.today().weekday()])


#### 按周遍历月份

`Calendar.monthdatescalendar()` 会返回完整周矩阵，矩阵中可能包含上个月或下个月的日期。这个结构很适合渲染日历组件或做按周统计。


In [ ]:
import calendar as calendar_tools
from datetime import date


def month_weeks(year: int, month: int, first_weekday: int = calendar_tools.MONDAY) -> list[list[date]]:
    """按周生成某个月的日期矩阵。

    Args:
        year: 年份。
        month: 月份，范围是 1 到 12。
        first_weekday: 每周第一天，默认周一。

    Returns:
        由周组成的日期矩阵。
    """
    cal = calendar_tools.Calendar(firstweekday=first_weekday)
    return cal.monthdatescalendar(year, month)


for week in month_weeks(2026, 5):
    print([day.isoformat() for day in week])


#### 工作日统计

`itermonthdates()` 适合按日期流式遍历一个月。统计工作日时要过滤掉矩阵中属于相邻月份的日期，再排除周末和节假日。


In [ ]:
import calendar as business_calendar
from datetime import date


def business_days(year: int, month: int, holidays: set[date] | None = None) -> list[date]:
    """统计某个月的工作日。

    Args:
        year: 年份。
        month: 月份，范围是 1 到 12。
        holidays: 需要排除的节假日集合。

    Returns:
        当月所有工作日。
    """
    holidays = holidays or set()
    cal = business_calendar.Calendar(firstweekday=business_calendar.MONDAY)
    days: list[date] = []

    for day in cal.itermonthdates(year, month):
        if day.month != month:
            continue
        if day.weekday() >= 5:
            continue
        if day in holidays:
            continue
        days.append(day)

    return days


may_holidays = {date(2026, 5, 1)}
workdays = business_days(2026, 5, may_holidays)
print("Business days:", len(workdays))
print("First five:", workdays[:5])


#### 文本和 HTML 输出

`TextCalendar` 适合命令行输出，`HTMLCalendar` 可以快速生成简单 HTML 表格。生产环境里通常会把这些数据喂给模板或前端组件，而不是直接使用默认样式。


In [ ]:
import calendar as html_calendar

text_calendar = html_calendar.TextCalendar(firstweekday=html_calendar.SUNDAY)
html_month = html_calendar.HTMLCalendar(firstweekday=html_calendar.SUNDAY).formatmonth(2026, 5)

print(text_calendar.formatmonth(2026, 5))
print(html_month[:200])


### 数值工具选择

| 工具 | 适合场景 | 避免场景 |
| --- | --- | --- |
| `math` | 实数数学、几何、组合数、普通统计 | 金额和强精度小数 |
| `cmath` | 复数、极坐标、复平面旋转 | 只需要实数结果的普通计算 |
| `decimal` | 金额、结算、明确舍入规则 | 大规模科学计算 |
| `fractions` | 比例、分数、精确有理数 | 分母可能快速膨胀的大量计算 |
